In [0]:
# ===========================================================================
# Notebook  : bronze_events
# Location  : /HGI-Lakehouse-Pipeline/02_Bronze_Layer/bronze_events
# Purpose   : BIGQUERY Event → bronze.{customer_id}.events_raw
# # BQ events: stored as events_raw (NOT crm_events)
# Serverless : Set max_files_per_trigger=10 for Serverless 2X-Small testing
# Job Compute: Set max_files_per_trigger=1000 for production
# ===========================================================================

In [0]:
# CELL 1 ── Load shared config
# In Databricks: uncomment the two %run lines below.
# Each %run MUST be in its own cell (magic commands require their own cell).
%run ../config/pipeline_config.py


In [0]:
%run ./bronze_common.py

In [0]:
# CELL 2 ── Widgets
dbutils.widgets.text("customer_id",           "")
dbutils.widgets.text("load_type",             "")   # historical | incremental
dbutils.widgets.text("max_files_per_trigger", "")
# Serverless tip: override max_files_per_trigger to '10' on Serverless 2X-Small

customer_id           = dbutils.widgets.get("customer_id").strip().lower()
load_type             = dbutils.widgets.get("load_type").strip().lower()
max_files_per_trigger = int(dbutils.widgets.get("max_files_per_trigger"))

In [0]:
# CELL 3 ── Object-specific constants
source_system  = "bigquery"
object_name    = "events"
sf_object_name = "Event"
# Composite ID will be: bigquery_Event_Id_{SalesforceId}
tenant_id = tenant_id_from_customer(customer_id)   # helper from pipeline_config


In [0]:
# CELL 4 ── Path resolution  (all helpers from pipeline_config)
landing_zone_path = landing_path(source_system, customer_id, object_name, load_type)
checkpoint_loc    = checkpoint_path("bronze", source_system, customer_id, object_name)
schema_loc        = checkpoint_loc + "_schema/"
table_full_name   = bronze_table(customer_id, object_name)
# table_full_name = bronze.{customer_id}.events_raw

print(f"=== Bronze: BIGQUERY Event ===")
print(f"  Customer   : {customer_id}  (tenant={tenant_id})")
print(f"  Load type  : {load_type}")
print(f"  Landing    : {landing_zone_path}")
print(f"  Target     : {table_full_name}")
print(f"  Batch size : {max_files_per_trigger} files/trigger")

In [0]:
# CELL 5 ── Create bronze schema + table
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {bronze_catalog}.{customer_id}")
create_bronze_table(table_full_name, object_name)  # uses BRONZE_DDL_COLUMNS['events']

In [0]:
# CELL 6 ── Run pipeline
run_with_retry(start_ingestion)

In [0]:
# CELL 7 ── Post-load maintenance
print("Running OPTIMIZE + VACUUM ...")
spark.sql(f"OPTIMIZE {table_full_name}")
spark.sql(f"VACUUM {table_full_name} RETAIN 168 HOURS")
print(f"Bronze load complete: {table_full_name}")
dbutils.notebook.exit("Success")